### 0. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

os.chdir("../")
from scripts import utils
from pathlib import Path
import matplotlib.gridspec as gridspec
from tqdm.auto import tqdm

In [2]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
from mlxtend.evaluate import feature_importance_permutation
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
from sklearn.utils.estimator_checks import check_estimator
from mlxtend.feature_selection import (
    SequentialFeatureSelector,
)
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
import matplotlib.ticker as ticker
import distclassipy as dcpy

In [4]:
epsilon = np.finfo(np.float32).eps

In [5]:
with open("settings.txt") as f:
    settings_dict = json.load(f)
seed_val = settings_dict["seed_choice"]
np.random.seed(seed_val)
sns_dict = settings_dict["sns_dict"]
sns.set_theme(**sns_dict)

In [6]:
unique_metrics = ['euclidean',
 'braycurtis',
 'canberra',
 'cityblock',
 'chebyshev',
 'clark',
 'correlation',
 'cosine',
 'hellinger',
 'jaccard',
 'lorentzian',
 # 'marylandbridge',
 'meehl',
 'motyka',
 'soergel',
 'wave_hedges',
 'kulczynski',
 # 'add_chisq'
                 ]


final_features = [
    "SPM_A_Y",
    "Multiband_period",
    "r-i",
    "Harmonics_phase_4_i",
    "Harmonics_phase_2_r",
    "Power_rate_4",
]

### 1. Data Prep
- Make a knowns/inlier dataset with 4 classes (CEP, RR, DSCT and EB): ```X_knowns_df``` and ```y_knowns_df```
- Make an unknowns/outlier/anomaly dataset with other classes: ```X_anom_df``` and ```y_anom_df```

An ```X_all_df``` and ```y_all_df``` contains both of these.

In [7]:
knowns = pd.read_parquet("data/reduced_balancedfeatures_LATEST.parquet")
knowns = knowns.sample(frac=1) #shuffle

y_knowns_df = knowns["class"]
X_knowns_df = knowns.loc[:, final_features]


unknowns = pd.read_parquet("data/otherclassobjs_features.parquet")
unknowns.index.name = "snid"
unknowns_lc_df = pd.read_parquet("data/otherclassobjs.parquet")
unknowns_lc_df.index.name = "snid"

unknowns_lc_df=unknowns_lc_df[~unknowns_lc_df["class"].isin(['d-Sct', 'Cepheid', 'EB', 'RRL'])]
unknowns=unknowns.loc[unknowns_lc_df.index]


unknowns_lc_df = unknowns_lc_df.loc[unknowns.index]

assert (unknowns.index == unknowns_lc_df.index).all()


X_anom_df = unknowns.loc[:, X_knowns_df.columns].dropna()
X_anom_df = X_anom_df.drop(np.intersect1d(X_anom_df.index, X_knowns_df.index))
y_anom_df = unknowns_lc_df.loc[X_anom_df.index]["class"]
X_anom_df=X_anom_df.loc[y_anom_df.index]

In [8]:
X_all_df = pd.concat([X_knowns_df, X_anom_df]).sample(frac=1)
y_all_df = pd.concat([y_knowns_df, y_anom_df]).loc[X_all_df.index]

### 2. Distance-Anomalies

In [9]:
from distclassipy.anomaly import DistanceAnomaly

```bash
cluster_agg : {'min', 'median'}, default='min'
    The aggregation method for distances to different class centroids for a
    single metric.
    - 'min': An object's distance is its distance to the *nearest* known class.
    - 'median': A more robust measure of an object's typical distance to all classes.

metric_agg : {'median', 'mean', 'min', 'p25'}, default='median'
    The method to aggregate scores from the ensemble of metrics.
    - 'median': The median of scores across all metrics. Robust to outlier metrics.
    - 'mean': The mean of scores.
    - 'min': The minimum score across all metrics.
    - 'percentile_25': The 25th percentile of scores.
```

In [10]:
ad_model = DistanceAnomaly(
    cluster_agg='min', # options:
    metric_agg='median',  # 'min' is equivalent to a 0th percentile
    metrics=unique_metrics
)

ad_model.fit(X_knowns_df.values, y_knowns_df.values)

anomaly_scores = ad_model.decision_function(X_all_df.values)

results_df = pd.DataFrame(index=X_all_df.index)
results_df['dcpy_score'] = anomaly_scores
results_df['class'] = y_all_df
results_df['status'] = results_df['class'].apply(
    lambda x: 'normal' if x in ["CEP", "DSCT", "EB", "RRL"] else 'anomalous'
)

print("Top anomalies")
results_df.sort_values("dcpy_score", ascending=False).head(20)

Top anomalies


,dcpy_score,class,status
snid,,,
10771496,4.176774,KN_K17,anomalous
88253880,3.986333,SNIa-91bg,anomalous
46466259,3.929110,dwarf-nova,anomalous
141445943,3.859973,ILOT,anomalous
96837672,3.821193,CEP,normal
37734415,3.738783,dwarf-nova,anomalous
37691298,3.662364,SNII-Templates,anomalous
75421831,3.656606,dwarf-nova,anomalous
99141506,3.492102,dwarf-nova,anomalous


In [11]:
ad_model = DistanceAnomaly(
    cluster_agg='median', # options:
    metric_agg='median',  # 'min' is equivalent to a 0th percentile
    metrics=unique_metrics
)

ad_model.fit(X_knowns_df.values, y_knowns_df.values)

anomaly_scores = ad_model.decision_function(X_all_df.values)

results_df = pd.DataFrame(index=X_all_df.index)
results_df['dcpy_score'] = anomaly_scores
results_df['class'] = y_all_df
results_df['status'] = results_df['class'].apply(
    lambda x: 'normal' if x in ["CEP", "DSCT", "EB", "RRL"] else 'anomalous'
)

print("Top anomalies")
results_df.sort_values("dcpy_score", ascending=False).head(20)

Top anomalies


,dcpy_score,class,status
snid,,,
73799575,5.445418,CEP,normal
103308635,5.171682,CEP,normal
59245585,5.169616,CEP,normal
108559610,4.954337,CEP,normal
17960110,4.896997,CEP,normal
126601452,4.849030,CEP,normal
4183976,4.830998,CEP,normal
130723508,4.802635,CEP,normal
81517825,4.788574,CEP,normal


In [12]:
ad_model = DistanceAnomaly(
    cluster_agg='min', # options:
    metric_agg='min',  # 'min' is equivalent to a 0th percentile
    metrics=unique_metrics
)

ad_model.fit(X_knowns_df.values, y_knowns_df.values)

anomaly_scores = ad_model.decision_function(X_all_df.values)

results_df = pd.DataFrame(index=X_all_df.index)
results_df['dcpy_score'] = anomaly_scores
results_df['class'] = y_all_df
results_df['status'] = results_df['class'].apply(
    lambda x: 'normal' if x in ["CEP", "DSCT", "EB", "RRL"] else 'anomalous'
)

print("Top anomalies")
results_df.sort_values("dcpy_score", ascending=False).head(20)

Top anomalies


,dcpy_score,class,status
snid,,,
88253880,0.772094,SNIa-91bg,anomalous
141445943,0.708816,ILOT,anomalous
37734415,0.703729,dwarf-nova,anomalous
140819459,0.700631,dwarf-nova,anomalous
118315745,0.695961,uLens-Single-GenLens,anomalous
114522598,0.683073,uLens-Single-GenLens,anomalous
103313063,0.679649,SNIa-91bg,anomalous
99141506,0.675586,dwarf-nova,anomalous
43265685,0.671533,PISN-STELLA_HYDROGENIC,anomalous
